In [1]:
import json
from pathlib import Path

In [2]:
#pip install torch

In [3]:
#pip install protobuf

In [4]:
#pip install langchain_openai langchain_chroma langchain_huggingface langchain_community langchain_text_splitters

In [5]:
#pip install tiktoken

In [6]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

In [8]:
#pip install --upgrade numexpr bottleneck

In [9]:
## The below uninstallations and installations are done as per ChatGPT advise
# I had an error "ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject"
# while running "from langchain_community.document_loaders import DirectoryLoader, TextLoader"

In [10]:
#pip uninstall numpy -y

In [11]:
#pip uninstall pandas scikit-learn -y

In [12]:
#pip install numpy==1.26.4

In [13]:
#pip install pandas scikit-learn

In [14]:
# pip install --upgrade huggingface_hub

In [15]:
# pip install transformers==4.57.6

In [16]:
# pip install --upgrade datasets==3.6.0

In [17]:
# This is done due to a warning during datasets library installation
#pip install --upgrade fsspec

In [18]:
entire_knowledge_base = ""

In [19]:
knowledge_base_path = "knowledge-base/**/*.json"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Total no. of files: {len(files)}")

for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        pages = json.load(f)['pages']
        for page in pages:
            entire_knowledge_base += page["text"]
        entire_knowledge_base += '\n\n'
        
print(f"Total characters: {len(entire_knowledge_base):,}")

Total no. of files: 32
Total characters: 1,835,607


In [20]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [21]:
load_dotenv(override=True)

True

In [22]:
openai_api_key = os.getenv("OPENAI_API_KEY")

In [23]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
print(f"Total tokens for {MODEL}: {len(tokens):,}")

Total tokens for gpt-4.1-nano: 399,433


In [24]:
def make_metadata_func(full_document):
    pass

In [25]:
def my_metadata_func(record: dict, base_metadata: dict) -> dict:
    print("Records: ")
    print(record)
    print("Base Metadata")
    print(base_metadata)
    print()
#     base_metadata["category"] = record.get("category")
#     base_metadata["policy_name"] = record.get("policy_name")
    return base_metadata

In [26]:
def get_pages(json_doc):
    for element in json_doc:
        if isinstance(element, list):
            return element

In [27]:
def load_json_with_root(file_path):
    with open(file_path, 'r') as f:
        full_data = json.load(f)
        policy_name = full_data.get("policy_name", "unknown")
        category = full_data.get("category", "unknown")
        source = full_data.get("source_path", "unknown")
        
    def metadata_func(record: dict, base_metadata: dict) -> dict:
        base_metadata["policy_name"] = policy_name
        base_metadata["category"] = category
        base_metadata["doc_type"] = record.get("page_type", "unknown")
        base_metadata['source'] = source
        return base_metadata
        
    return JSONLoader(
        file_path=file_path,
        jq_schema='.pages[]',
        content_key='text',
        metadata_func=metadata_func
    )

In [28]:
# folders = glob.glob("knowledge-base/*")

# documents = []
# for folder in folders:
#     loader = DirectoryLoader(folder, glob="**/*.json", loader_cls=JSONLoader,\
#                             loader_kwargs={'jq_schema': '.pages[]', \
#                                           'content_key': 'text', \
#                                            'metadata_func': my_metadata_func\
#                                           })
#     #loader = JSONLoader(folder, )
#     folder_docs = loader.load()
#     for doc in folder_docs:
#         documents.append(doc)

# print(f"Loaded {len(documents)} documents")

In [29]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    loader = DirectoryLoader(folder, glob="**/*.json", loader_cls=load_json_with_root)
    #loader = JSONLoader(folder, )
    folder_docs = loader.load()
    for doc in folder_docs:
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 544 documents


In [30]:
# documents[0]

In [ ]:
# folders = glob.glob("lic_policies/*")

# for folder in folders:
#     loader = DirectoryLoader(folder, glob="**/*.pdf", loader_cls=PyPDFLoader)
#     #loader = JSONLoader(folder, )
#     folder_docs = loader.load()
#     print(folder_docs)

In [ ]:
#pip install jq

In [ ]:
#pip install pypdf

In [ ]:
text_splitters = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitters.split_documents(documents)
len(chunks)

In [ ]:
chunks[100]

In [ ]:
embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

In [ ]:
# pip install sentence-transformers

In [ ]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
    
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

In [ ]:
print(f"vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
#retriever.invoke("How many plans are there for endowment?")

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the LIC (Life Insurance Corporation of India).
You are chatting with a user about LIC's insurance products only.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("How many plans are there for endowment?", [])

In [ ]:
#pip install gradio

In [ ]:
gr.ChatInterface(answer_question).launch()